# 디몰비전 QLoRA 파인튜닝 (Colab Pro+ / A100)

기획서 §4.2·§6.2. 외부 AI API 무종속 — 오픈웨이트 Qwen3-VL + 로컬 QLoRA.

파이프라인: **CODD 공개데이터 다운로드 → prepare_data → QLoRA 학습 → 평가 게이트**.
석면(ROOFING_SLATE)은 공개 라벨이 없어 양평 자체수집 데이터를 별도로 합류시켜야 함(§V2).

## 0. GPU 확인 (런타임 → 런타임 유형 변경 → A100)

In [ ]:
!nvidia-smi

## 1. 의존성 설치

In [ ]:
!pip -q install "transformers>=4.46" peft trl bitsandbytes datasets accelerate pyyaml pillow

## 2. 리포지토리 배치

`apps/vision-api` 를 Colab 세션에 올린다. (a) 깃 클론 또는 (b) 드라이브 마운트 후 경로 지정.

In [ ]:
import os
# 예: 구글드라이브에 업로드한 경우
# from google.colab import drive; drive.mount('/content/drive')
# REPO = '/content/drive/MyDrive/dimollink/apps/vision-api'
REPO = '/content/vision-api'   # 깃 클론 시
os.chdir(REPO)
print('cwd:', os.getcwd())

## 3. CODD 공개 데이터 다운로드 (CC BY 4.0)

Mendeley Data: https://data.mendeley.com/datasets/wds85kt64j/3 — 3,129장 / 10클래스 / COCO.
아래는 자동 다운로드 예시(수동 다운로드 링크가 바뀔 수 있어 실패 시 브라우저로 받아 `data/CODD/` 에 배치).

In [ ]:
import os, urllib.request, zipfile
os.makedirs('data/CODD', exist_ok=True)
# Mendeley 는 직접 링크가 세션별로 다를 수 있음. 실패하면 수동 업로드로 대체.
# 기대 구조: data/CODD/images/*.jpg , data/CODD/annotations.json (COCO)
print('data/CODD 준비:', os.listdir('data/CODD') if os.path.isdir('data/CODD') else '비어있음')

## 4. 학습 JSONL 생성 (공개 클래스 → 우리 12종 매핑)

In [ ]:
!python scripts/prepare_data.py --format coco --dataset codd \
    --coco data/CODD/annotations.json --images data/CODD/images \
    --out data/train_recognition.jsonl

### (선택) 양평 자체수집 석면 데이터 합류

CODD 엔 석면이 없다. 자체 라벨(폴더 형식)을 같은 스키마로 변환해 이어붙인다.

In [ ]:
# !python scripts/prepare_data.py --format folder --dataset roboflow_construction \
#     --images data/yangpyeong_slate --out data/train_slate.jsonl
# !cat data/train_slate.jsonl >> data/train_recognition.jsonl   # 합류

## 5. 배선 검증(dry-run) — 데이터·config·대장 레코드 확인

In [ ]:
!python training/train.py --config training/qlora_config.yaml --task recognition --dry-run

## 6. QLoRA 학습 루프 (A100)

Qwen3-VL 을 4bit 로드 + LoRA. 각 예시는 (이미지 → 자재 JSON) 지도학습.

In [ ]:
import json, yaml, torch
from PIL import Image
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (AutoProcessor, AutoModelForVision2Seq,
                          BitsAndBytesConfig, TrainingArguments, Trainer)

cfg = yaml.safe_load(open('training/qlora_config.yaml'))
rows = [json.loads(l) for l in open('data/train_recognition.jsonl') if l.strip()]
print('examples:', len(rows))

SYS = ('너는 철거현장 자재 인식기다. 이미지의 해체 자재를 우리 12종 코드로만 답한다. '
       'JSON: {"materials":[코드,...]}. 석면 의심(ROOFING_SLATE)은 "확정"이 아닌 "의심"으로만.')

def to_msg(r):
    tgt = json.dumps({"materials": r["materials"]}, ensure_ascii=False)
    return {"image": r["image"], "prompt": SYS, "answer": tgt}
ds = Dataset.from_list([to_msg(r) for r in rows])

In [ ]:
q = cfg['quantization']
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=q['bnb_4bit_quant_type'],
                         bnb_4bit_compute_dtype=getattr(torch, q['bnb_4bit_compute_dtype']))
proc = AutoProcessor.from_pretrained(cfg['base_model'], trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(cfg['base_model'], quantization_config=bnb,
                                               device_map='auto', trust_remote_code=True)
lc = cfg['lora']
model = get_peft_model(model, LoraConfig(r=lc['r'], lora_alpha=lc['alpha'],
        lora_dropout=lc['dropout'], target_modules=lc['target_modules'], task_type='CAUSAL_LM'))
model.print_trainable_parameters()

In [ ]:
def collate(batch):
    texts, images = [], []
    for b in batch:
        msgs = [{"role":"system","content":b["prompt"]},
                {"role":"user","content":[{"type":"image"},{"type":"text","text":"이 현장의 자재는?"}]},
                {"role":"assistant","content":b["answer"]}]
        texts.append(proc.apply_chat_template(msgs, tokenize=False))
        images.append(Image.open(b["image"]).convert("RGB"))
    enc = proc(text=texts, images=images, return_tensors="pt", padding=True)
    enc["labels"] = enc["input_ids"].clone()
    return enc

t = cfg['train']
args = TrainingArguments(output_dir=cfg['tasks']['recognition']['output_adapter'],
    num_train_epochs=t['epochs'], per_device_train_batch_size=t['batch_size'],
    gradient_accumulation_steps=t['grad_accum'], learning_rate=t['lr'],
    warmup_ratio=t['warmup_ratio'], seed=t['seed'], bf16=True, logging_steps=10, save_strategy='epoch')
Trainer(model=model, args=args, train_dataset=ds, data_collator=collate).train()
model.save_pretrained(cfg['tasks']['recognition']['output_adapter'])
print('어댑터 저장:', cfg['tasks']['recognition']['output_adapter'])

## 7. 평가 게이트 (§V3 종료 기준)

학습셋과 분리된 평가셋으로 채점. 주요 5종 재현율 ≥ 0.85, 석면 의심 ≥ 0.95 미달 시 exit 1.
합격한 어댑터만 `model_versions` 대장 기록 후 배포(`POST /admin/reload`).

In [ ]:
!python scripts/eval.py --evalset eval_samples/sample_evalset.jsonl
# 실제로는 30현장 평가셋(추론 결과 포함)으로 교체